# Laboratorio Práctico de Machine Learning — Dry Bean Dataset
**CRISP-DM + TDSP + Scrum ML**

---

## Equipo Scrum ML

| Rol | Responsable |
|-----|-------------|
| Product Owner | Carlos Garzón |
| Scrum Master | Carlos Garzón |
| Data Engineer / Analyst | Jhony Posada |
| ML Engineer | Jhony Posada |

**Sprint activo:** Sprint 1 — Comprensión y preparación  
**Fecha:** Mayo 2026  
**Dataset:** Dry Bean Dataset, UCI ML Repository (id=602)

---
## CRISP-DM Fase 1: Comprensión del Negocio

**Problema de negocio:** Una empresa agrícola desea automatizar la clasificación de variedades de granos de frijol seco para mejorar el control de calidad. Actualmente parte del proceso es manual.

**Objetivo del modelo:** Predecir la variedad de frijol (`Class`) a partir de 16 características geométricas extraídas mediante visión por computador.

**Tipo de problema:** Clasificación multiclase (7 clases).

**Criterio de éxito:** F1-macro ≥ 0.90 en el conjunto de prueba.

**Conexión Scrum ML:** Esta comprensión fue definida por el *Product Owner* como historia de usuario PB-01.

---
## Paso 1 — Sprint 1: Importar librerías

**Responsable:** Data Engineer  
**Tarea backlog:** PB-01 — Descargar dataset Dry Bean

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Para entornos sin display
import os

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay
)
import joblib

# Crear directorios si no existen
os.makedirs('../outputs/models', exist_ok=True)
os.makedirs('../outputs/reports', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('✅ Librerías importadas correctamente')
print(f'   pandas {pd.__version__} | numpy {np.__version__}')

✅ Librerías importadas correctamente
   pandas 3.0.2 | numpy 2.4.4


---
## CRISP-DM Fase 2: Comprensión de Datos — Paso 2: Cargar dataset desde UCI

In [2]:
# Descargar el dataset Dry Bean desde UCI ML Repository
# Identificador oficial: id=602
dry_bean = fetch_ucirepo(id=602)

X = dry_bean.data.features   # Variables predictoras (16 características geométricas)
y = dry_bean.data.targets    # Variable objetivo: Class (variedad de frijol)

df = pd.concat([X, y], axis=1)

print('✅ Dataset cargado exitosamente')
print(f'   Instancias: {df.shape[0]:,} | Columnas: {df.shape[1]}')
df.head()

✅ Dataset cargado exitosamente
   Instancias: 13,611 | Columnas: 17


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


---
## Paso 3: Comprensión de datos (CRISP-DM Fase 2)
**Tarea backlog:** PB-02 — Analizar calidad de datos  
**Responsable:** Data Analyst

In [3]:
print('=== DIMENSIONES ===')
print(f'Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')

print('\n=== TIPOS DE DATOS ===')
print(df.dtypes)

print('\n=== ESTADÍSTICAS DESCRIPTIVAS ===')
df.describe().round(3)

=== DIMENSIONES ===
Filas: 13,611  |  Columnas: 17

=== TIPOS DE DATOS ===
Area                 int64
Perimeter          float64
MajorAxisLength    float64
MinorAxisLength    float64
AspectRatio        float64
Eccentricity       float64
ConvexArea           int64
EquivDiameter      float64
Extent             float64
Solidity           float64
Roundness          float64
Compactness        float64
ShapeFactor1       float64
ShapeFactor2       float64
ShapeFactor3       float64
ShapeFactor4       float64
Class                  str
dtype: object

=== ESTADÍSTICAS DESCRIPTIVAS ===


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4
count,13611.000,13611.000,13611.000,13611.000,13611.000,13611.000,13611.000,13611.000,13611.000,13611.000,13611.000,13611.000,13611.000,13611.000,13611.000,13611.000
mean,53048.285,855.283,320.142,202.271,1.583,0.751,53768.200,253.064,0.750,0.987,0.873,0.800,0.007,0.002,0.644,0.995
std,29324.096,214.290,85.694,44.970,0.247,0.092,29774.916,59.177,0.049,0.005,0.060,0.062,0.001,0.001,0.099,0.004
min,20420.000,524.736,183.601,122.513,1.025,0.219,20684.000,161.244,0.555,0.919,0.490,0.641,0.003,0.001,0.410,0.948
25%,36328.000,703.524,253.304,175.848,1.432,0.716,36714.500,215.068,0.719,0.986,0.832,0.762,0.006,0.001,0.581,0.994
50%,44652.000,794.941,296.883,192.432,1.551,0.764,45178.000,238.438,0.760,0.988,0.883,0.801,0.007,0.002,0.642,0.996
75%,61332.000,977.213,376.495,217.032,1.707,0.810,62294.000,279.446,0.787,0.990,0.917,0.834,0.007,0.002,0.696,0.998
max,254616.000,1985.370,738.860,460.198,2.430,0.911,263261.000,569.374,0.866,0.995,0.991,0.987,0.010,0.004,0.975,1.000


### Respuestas a actividad del estudiante (Paso 3)

1. **¿Cuántas columnas tiene el dataset?** 17 columnas (16 features + 1 objetivo)
2. **¿Cuál es la variable objetivo?** `Class` — variedad de frijol (SEKER, BARBUNYA, BOMBAY, CALI, HOROZ, SIRA, DERMASON)
3. **¿El problema es de clasificación o regresión?** **Clasificación multiclase** — el target es categórico con 7 clases discretas.

---
## Paso 4: Revisar calidad de datos — Nulos y duplicados

In [4]:
# Valores nulos por columna
missing_values = df.isna().sum()
duplicated_rows = df.duplicated().sum()

print('=== VALORES NULOS POR COLUMNA ===')
print(missing_values)
print(f'\nTotal nulos: {missing_values.sum()}')
print(f'Filas duplicadas: {duplicated_rows}')

# Decisión técnica: eliminar duplicados si existen
if duplicated_rows > 0:
    df = df.drop_duplicates()
    print(f'\n⚠️  Se eliminaron {duplicated_rows} filas duplicadas')
    print(f'   Nuevo tamaño: {df.shape[0]:,} filas')
else:
    print('\n✅ No hay duplicados — dataset íntegro')

=== VALORES NULOS POR COLUMNA ===
Area               0
Perimeter          0
MajorAxisLength    0
MinorAxisLength    0
AspectRatio        0
Eccentricity       0
ConvexArea         0
EquivDiameter      0
Extent             0
Solidity           0
Roundness          0
Compactness        0
ShapeFactor1       0
ShapeFactor2       0
ShapeFactor3       0
ShapeFactor4       0
Class              0
dtype: int64

Total nulos: 0
Filas duplicadas: 68

⚠️  Se eliminaron 68 filas duplicadas
   Nuevo tamaño: 13,543 filas


---
## Paso 5: Distribución de la variable objetivo (clases)

In [5]:
# Análisis de distribución de clases
class_counts = df['Class'].value_counts()
class_pct = (class_counts / len(df) * 100).round(1)

print('=== DISTRIBUCIÓN DE CLASES ===')
dist_df = pd.DataFrame({'Cantidad': class_counts, 'Porcentaje (%)': class_pct})
print(dist_df)

# Gráfica de barras
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#c0392b', '#e67e22', '#f1c40f', '#27ae60', '#2980b9', '#8e44ad', '#2c3e50']
class_counts.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.set_title('Distribución de clases — Dry Bean Dataset', fontsize=13, fontweight='bold')
ax.set_xlabel('Variedad de frijol', fontsize=11)
ax.set_ylabel('Cantidad de registros', fontsize=11)
ax.tick_params(axis='x', rotation=45)

for i, (v, p) in enumerate(zip(class_counts, class_pct)):
    ax.text(i, v + 30, f'{p}%', ha='center', fontsize=9, color='#333')

plt.tight_layout()
plt.savefig('../outputs/reports/distribucion_clases.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📊 Gráfica guardada en outputs/reports/distribucion_clases.png')

=== DISTRIBUCIÓN DE CLASES ===
          Cantidad  Porcentaje (%)
Class                             
DERMASON      3546            26.2
SIRA          2636            19.5
SEKER         2027            15.0
HOROZ         1860            13.7
CALI          1630            12.0
BARBUNYA      1322             9.8
BOMBAY         522             3.9

📊 Gráfica guardada en outputs/reports/distribucion_clases.png


C:\Users\cdgg4\AppData\Local\Temp\ipykernel_31336\2820604348.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Respuesta: ¿Las clases están balanceadas?

**No.** Existe desbalance moderado. DERMASON representa ~26% de las instancias mientras BOMBAY apenas ~3.9%. Este desbalance implica:

- El **accuracy puede ser engañoso** — un clasificador que siempre prediga la clase mayoritaria puede tener accuracy aceptable sin aprender nada útil.
- Usar **F1-macro** como métrica principal compensa el desbalance al tratar todas las clases con igual peso.
- En el Random Forest se usará `class_weight='balanced'` para ajustar los pesos de cada clase.

---
## Paso 6: Preparación de datos (CRISP-DM Fase 3)
Separación de variables predictoras/objetivo y split train/test

In [6]:
# Separar features y target
X = df.drop(columns='Class')
y = df['Class']

print(f'X shape: {X.shape}  |  y shape: {y.shape}')
print(f'Features: {X.columns.tolist()}')

X shape: (13543, 16)  |  y shape: (13543,)
Features: ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRatio', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'Roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4']


In [7]:
# Split estratificado: 80% train, 20% test
# random_state=42 → reproducibilidad (TDSP best practice)
# stratify=y → mantiene proporciones de clases en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('✅ Split completado')
print(f'   Train: {X_train.shape[0]:,} muestras ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'   Test:  {X_test.shape[0]:,} muestras ({X_test.shape[0]/len(X)*100:.0f}%)')

# Verificar proporciones de clases en train vs test
print('\n=== PROPORCIÓN DE CLASES EN TRAIN vs TEST ===')
train_dist = y_train.value_counts(normalize=True).round(3)
test_dist  = y_test.value_counts(normalize=True).round(3)
comp = pd.DataFrame({'Train': train_dist, 'Test': test_dist})
print(comp)

# Guardar datasets procesados
X_train.join(y_train).to_csv('../data/processed/train_set.csv', index=False)
X_test.join(y_test).to_csv('../data/processed/test_set.csv', index=False)
print('\n💾 Datos procesados guardados en data/processed/')

✅ Split completado
   Train: 10,834 muestras (80%)
   Test:  2,709 muestras (20%)

=== PROPORCIÓN DE CLASES EN TRAIN vs TEST ===
          Train   Test
Class                 
DERMASON  0.262  0.262
SIRA      0.195  0.195
SEKER     0.150  0.150
HOROZ     0.137  0.137
CALI      0.120  0.120
BARBUNYA  0.098  0.098
BOMBAY    0.039  0.038

💾 Datos procesados guardados en data/processed/


---
## ✅ Sprint 1 — Definition of Done

| Criterio | Estado |
|----------|--------|
| Dataset cargado desde UCI | ✅ |
| Análisis exploratorio básico | ✅ |
| Nulos y duplicados reportados | ✅ |
| Distribución de clases analizada | ✅ |
| Split train/test estratificado | ✅ |
| Datos procesados guardados | ✅ |

**Retrospectiva Sprint 1:** El dataset no tiene nulos. El mayor reto fue identificar el desbalance de clases y decidir la estrategia para manejarlo en el modelado.

---
# Sprint 2 — Modelado y Evaluación

**Objetivo del sprint:** Entrenar modelo baseline y modelo mejorado, comparar métricas, guardar el mejor modelo.

**Conexión CRISP-DM:** Fases 4 (Modelado) y 5 (Evaluación).

---
## Paso 7: Modelo Baseline — Regresión Logística
**Tarea backlog:** PB-03 — Entrenar Logistic Regression  
**Responsable:** ML Engineer

In [ ]:
# Pipeline: StandardScaler → LogisticRegression
# El escalado es necesario para regresión logística (sensible a escala)
baseline_model = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

baseline_model.fit(X_train, y_train)
print('✅ Modelo baseline entrenado (Logistic Regression)')

---
## Paso 8: Evaluar modelo baseline

In [ ]:
y_pred_baseline = baseline_model.predict(X_test)

accuracy_baseline = accuracy_score(y_test, y_pred_baseline)
f1_baseline       = f1_score(y_test, y_pred_baseline, average='macro')

print(f'=== BASELINE — Logistic Regression ===')
print(f'Accuracy : {accuracy_baseline:.4f} ({accuracy_baseline*100:.2f}%)')
print(f'F1-macro : {f1_baseline:.4f}')
print('\n=== REPORTE POR CLASE ===')
print(classification_report(y_test, y_pred_baseline))

### Respuesta: ¿Por qué F1-macro es más informativo que accuracy?

**Accuracy** da el mismo peso a cada instancia. Si DERMASON representa 26% del dataset y el modelo lo clasifica bien, el accuracy se infla aunque falle en BOMBAY (3.8%). 

**F1-macro** promedia el F1 de cada clase con **igual peso** independientemente del tamaño de la clase. Si el modelo falla en BOMBAY, ese error impacta igual que si falla en DERMASON. Es la métrica correcta para clasificación multiclase con desbalance.

---
## Paso 9: Modelo Mejorado — Random Forest
**Tarea backlog:** PB-04 — Entrenar Random Forest  
**Responsable:** ML Engineer

In [ ]:
# Random Forest no requiere escalado (basado en árboles)
# class_weight='balanced' compensa el desbalance entre clases
forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

forest_model.fit(X_train, y_train)
print('✅ Modelo mejorado entrenado (Random Forest, 200 árboles)')

---
## Paso 10: Evaluar Random Forest

In [ ]:
y_pred_forest = forest_model.predict(X_test)

accuracy_forest = accuracy_score(y_test, y_pred_forest)
f1_forest       = f1_score(y_test, y_pred_forest, average='macro')

print(f'=== MODELO MEJORADO — Random Forest ===')
print(f'Accuracy : {accuracy_forest:.4f} ({accuracy_forest*100:.2f}%)')
print(f'F1-macro : {f1_forest:.4f}')
print('\n=== REPORTE POR CLASE ===')
print(classification_report(y_test, y_pred_forest))

---
## Paso 11: Comparar modelos (CRISP-DM Fase 5 — Evaluación)
**Tarea backlog:** PB-05 — Comparar métricas

In [ ]:
results = pd.DataFrame({
    'modelo'  : ['Logistic Regression (baseline)', 'Random Forest'],
    'accuracy': [accuracy_baseline, accuracy_forest],
    'f1_macro': [f1_baseline, f1_forest]
})

print('=== COMPARACIÓN DE MODELOS ===')
print(results.to_string(index=False))

# Decisión CRISP-DM
mejor = 'Random Forest' if f1_forest >= f1_baseline else 'Logistic Regression'
mejora = abs(f1_forest - f1_baseline)
print(f'\n🏆 Modelo seleccionado: {mejor}')
print(f'   Mejora en F1-macro: +{mejora:.4f}')

---
## Paso 12: Matriz de confusión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Baseline
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_baseline,
    xticks_rotation=45, ax=axes[0],
    colorbar=False
)
axes[0].set_title(f'Logistic Regression\nAcc={accuracy_baseline:.3f} | F1={f1_baseline:.3f}',
                  fontsize=11, fontweight='bold')

# Random Forest
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_forest,
    xticks_rotation=45, ax=axes[1],
    colorbar=False
)
axes[1].set_title(f'Random Forest\nAcc={accuracy_forest:.3f} | F1={f1_forest:.3f}',
                  fontsize=11, fontweight='bold')

plt.suptitle('Matrices de Confusión — Dry Bean Dataset', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/reports/matrices_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Guardado en outputs/reports/matrices_confusion.png')

---
## Paso 13: Importancia de variables

In [ ]:
# Importancia de features según Random Forest
feature_importance = pd.DataFrame({
    'feature'   : X.columns,
    'importance': forest_model.feature_importances_
}).sort_values('importance', ascending=False)

print('=== TOP 10 VARIABLES MÁS IMPORTANTES ===')
print(feature_importance.head(10).to_string(index=False))

# Gráfica
fig, ax = plt.subplots(figsize=(10, 5))
top10 = feature_importance.head(10)
ax.bar(range(10), top10['importance'], color='#2980b9', edgecolor='white')
ax.set_xticks(range(10))
ax.set_xticklabels(top10['feature'], rotation=45, ha='right')
ax.set_title('Top 10 variables más importantes — Random Forest', fontsize=12, fontweight='bold')
ax.set_xlabel('Variable')
ax.set_ylabel('Importancia (Gini)')
plt.tight_layout()
plt.savefig('../outputs/reports/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Guardado en outputs/reports/feature_importance.png')

---
## ✅ Sprint 2 — Definition of Done

| Criterio | Estado |
|----------|--------|
| Modelo baseline (Logistic Regression) | ✅ |
| Modelo mejorado (Random Forest 200 árboles) | ✅ |
| Métricas accuracy y F1-macro calculadas | ✅ |
| Comparación de modelos documentada | ✅ |
| Matriz de confusión generada | ✅ |
| Importancia de variables analizada | ✅ |

---
# Sprint 3 — Despliegue y Documentación

**Objetivo:** Guardar modelo, crear función reutilizable, guardar predicciones CSV, validar reproducibilidad.

---
## Paso 14: Guardar el modelo final (TDSP — Versionado de artefactos)

In [ ]:
# Guardar con joblib — permite reproducibilidad y compartir el artefacto
model_path = '../outputs/models/random_forest_drybean.joblib'
joblib.dump(forest_model, model_path)
print(f'✅ Modelo guardado en: {model_path}')

# Verificación de integridad: cargar y predecir
loaded_model = joblib.load(model_path)
sample       = X_test.iloc[[0]]
prediction   = loaded_model.predict(sample)

print(f'\n=== VERIFICACIÓN DE CARGA ===')
print(f'Predicción: {prediction[0]}')
print(f'Valor real: {y_test.iloc[0]}')
print(f'Correcto:   {prediction[0] == y_test.iloc[0]}')

---
## Paso 15: Función de despliegue reutilizable (CRISP-DM Fase 6)

In [ ]:
def predict_bean_class(model, input_data):
    """
    Predice la variedad de frijol dadas sus características geométricas.
    
    Args:
        model:      modelo entrenado cargado con joblib
        input_data: DataFrame con 16 features geométricas
    
    Returns:
        str: variedad de frijol predicha
    """
    prediction = model.predict(input_data)
    return prediction[0]

# Demo
resultado = predict_bean_class(loaded_model, sample)
print(f'🫘 Variedad predicha: {resultado}')
print(f'   Clase real:        {y_test.iloc[0]}')

---
## Extensión C: Guardar predicciones en CSV

In [ ]:
predictions_df = X_test.copy()
predictions_df['real_class']      = y_test.values
predictions_df['predicted_class'] = y_pred_forest
predictions_df['correct']         = predictions_df['real_class'] == predictions_df['predicted_class']

csv_path = '../outputs/reports/predicciones_drybean.csv'
predictions_df.to_csv(csv_path, index=False)

error_rate = (~predictions_df['correct']).sum()
print(f'✅ Predicciones guardadas: {csv_path}')
print(f'   Total muestras: {len(predictions_df):,}')
print(f'   Correctas:      {predictions_df["correct"].sum():,}')
print(f'   Incorrectas:    {error_rate:,}')

---
## Extensión A: Validación cruzada para estabilidad del modelo

In [ ]:
print('Ejecutando cross-validation (5-fold) en Random Forest...')
cv_scores = cross_val_score(forest_model, X, y, cv=5, scoring='f1_macro', n_jobs=-1)
print(f'F1-macro por fold: {cv_scores.round(4)}')
print(f'Media: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

---
## Reflexiones finales (Preguntas del laboratorio)

1. **¿Qué fase de CRISP-DM fue más importante?** La comprensión de datos — identificar el desbalance de clases determinó toda la estrategia de modelado.

2. **¿Qué ventaja ofrece TDSP?** Estructura reproducible: cualquier persona puede clonar el repo, instalar `requirements.txt` y reproducir exactamente los mismos resultados.

3. **¿Cómo ayuda Scrum ML ante incertidumbre?** Permite entregar valor incremental. Si el baseline ya cumple el criterio de éxito, no se invierte más tiempo innecesariamente en modelos complejos.

4. **¿Por qué un modelo baseline?** Establece el piso de comparación. Sin baseline, no hay manera de saber si un modelo complejo realmente agrega valor.

5. **¿Qué métrica para escoger el mejor modelo?** F1-macro — por el desbalance de clases y porque el negocio necesita clasificar bien *todas* las variedades, no solo la mayoritaria.

6. **¿Qué hacer si una clase tiene muchos errores?** Revisar si hay suficientes datos de esa clase, considerar oversampling (SMOTE), ajustar `class_weight`, o añadir features discriminativas específicas para esa clase.

7. **¿Qué se necesita antes de producción?** Validación con datos frescos, definir SLA de latencia, monitoreo de drift de datos, pipeline de reentrenamiento, y validación del negocio con el Product Owner.

---
## ✅ Sprint 3 — Definition of Done completa

| Criterio DoD | Estado |
|-------------|--------|
| Notebook corre de inicio a fin sin errores | ✅ |
| Dataset descargado correctamente | ✅ |
| Nulos y duplicados reportados | ✅ |
| Modelo baseline entrenado | ✅ |
| Modelo alternativo entrenado | ✅ |
| Accuracy y F1-macro comparados | ✅ |
| Matriz de confusión incluida | ✅ |
| Modelo guardado con joblib | ✅ |
| Resultados explicados claramente | ✅ |
| CRISP-DM, TDSP y Scrum ML documentados | ✅ |